# Arquimedes QLoRA fine-tuning — GSM8K

Trains a LoRA adapter on top of `Qwen/Qwen2.5-1.5B-Instruct` so the
Archimedes tutor can fall back to a specialised math solver.

**Runtime**: Colab T4 / RunPod A10 / any GPU with >= 12 GB VRAM.
**Cost**: ~$2 on RunPod spot; free on Colab Pro.
**Wall clock**: ~2 h for 1 epoch over 5 000 GSM8K examples.

At the end the adapter is pushed to the HuggingFace Hub under
`HF_FINETUNED_REPO`, and the `solve_with_finetuned` tool can then load it.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets sentencepiece pyyaml

In [ ]:
import os, yaml, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

CONFIG = yaml.safe_load(open('config.yaml'))
BASE = CONFIG['base_model']
print('base model:', BASE)

In [ ]:
# ── 4-bit quantization config ────────────────────────────────────────
q = CONFIG['quantization']
bnb = BitsAndBytesConfig(
    load_in_4bit=q['load_in_4bit'],
    bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=getattr(torch, q['bnb_4bit_compute_dtype']),
)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)

In [ ]:
# ── LoRA adapter ─────────────────────────────────────────────────────
lora = CONFIG['lora']
peft_config = LoraConfig(
    r=lora['r'], lora_alpha=lora['alpha'], lora_dropout=lora['dropout'],
    target_modules=lora['target_modules'], bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# ── Dataset — GSM8K formatted as chat-style prompt+solution ──────────
ds_cfg = CONFIG['dataset']
template = ds_cfg['prompt_template']
train_ds = load_dataset(ds_cfg['name'], ds_cfg['config'], split=ds_cfg['train_split'])

def format_sample(ex):
    return {'text': template.format(question=ex['question']) + ex[ds_cfg['target_field']] + tok.eos_token}

train_ds = train_ds.map(format_sample)
print('train size:', len(train_ds))
print(train_ds[0]['text'][:400])

In [ ]:
# ── Trainer ──────────────────────────────────────────────────────────
tr = CONFIG['training']
output_dir = f"./adapters/{CONFIG['output_repo']}"
sft_cfg = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=tr['epochs'],
    per_device_train_batch_size=tr['per_device_batch_size'],
    gradient_accumulation_steps=tr['gradient_accumulation_steps'],
    learning_rate=tr['learning_rate'],
    lr_scheduler_type=tr['lr_scheduler_type'],
    warmup_ratio=tr['warmup_ratio'],
    max_seq_length=tr['max_seq_length'],
    bf16=tr['bf16'],
    gradient_checkpointing=tr['gradient_checkpointing'],
    logging_steps=tr['logging_steps'],
    save_steps=tr['save_steps'],
    dataset_text_field='text',
    report_to='none',
)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, args=sft_cfg)
trainer.train()
trainer.save_model(output_dir)

In [ ]:
# ── Sanity inference ─────────────────────────────────────────────────
from transformers import pipeline
pipe = pipeline('text-generation', model=model, tokenizer=tok)
prompt = template.format(question='What is 17 * 23?')
print(pipe(prompt, max_new_tokens=120, do_sample=False)[0]['generated_text'])

In [ ]:
# ── Push to Hub ──────────────────────────────────────────────────────
import huggingface_hub
huggingface_hub.login(token=os.environ.get('HF_TOKEN'))
repo_id = f"{os.environ['HF_USER']}/{CONFIG['output_repo']}"
trainer.push_to_hub(repo_id)
print(f'adapter pushed to: {repo_id}')
print('set HF_FINETUNED_REPO=' + repo_id + ' in the main app .env')